# Разработка A/B-тестирования и анализ результатов

Вы работаете в компании развлекательного приложения с «бесконечной» лентой коротких видео. Основная цель продукта — удерживать внимание пользователя.

Главный двигатель вашего приложения — это алгоритм рекомендаций. До недавнего времени лента формировалась на основе простых эвристик по схеме «популярное за день + новинки». Сейчас всё меняется: команда отдела Data Science разработала новый алгоритм ранжирования контента.

Команда DS утверждает, что новый алгоритм значимо повысит глубину просмотра `page_counter` и удержание. Ваша задача — спроектировать дизайн A/B-теста, рассчитать его параметры на исторических данных и вынести вердикт: стоит ли внедрять модель DS-команды для всех пользователей.

## Импорт необходимых библиотек и методов

Импортируйте библиотеки и функции, необходимые для работы с данными, визуализаций, расчёта размера выборки и проверки статистических гипотез.

Часть инструментов уже импортирована. Дополните ячейку ниже необходимыми импортами.

In [2]:
# Импортируйте pandas, ceil, matplotlib и seaborn
import pandas as pd
from math import ceil
import matplotlib.pyplot as plt
import seaborn as sns

# Импорт инструментов для расчёта мощности теста и эффекта
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Импорт функции z-теста для сравнения долей
from statsmodels.stats.proportion import proportions_ztest

## 1. Работа с историческими данными (EDA)

### 1.1 Загрузка исторических данных
Загрузите исторические данные о сессиях пользователей из файла `sessions_project_history.csv` и сохраните их в датафрейм `sessions_history`.

Затем:

- выведите общую информацию о датафрейме;
- выведите первые 5 строк датафрейма;
- при необходимости проведите предобработку данных.

Сделайте выводы о данных и проделанной предобработке.

In [3]:
# Загрузите данные и выполните парсинг колонок с датой и временем.
# Чтобы выполнить парсинг, укажите колонки в аргументе parse_dates
DATA_PATH = "https://code.s3.yandex.net/datasets/"

sessions_history = pd.read_csv(DATA_PATH + "sessions_project_history.csv",
                               parse_dates=["session_date", "session_start_ts", "install_date"])

In [5]:
# Выведите информацию о загруженных данных
display(sessions_history.info(), sessions_history.describe(include="all").T)

<class 'pandas.DataFrame'>
RangeIndex: 435924 entries, 0 to 435923
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   user_id            435924 non-null  str           
 1   session_id         435924 non-null  str           
 2   session_date       435924 non-null  datetime64[us]
 3   session_start_ts   435924 non-null  datetime64[us]
 4   install_date       435924 non-null  datetime64[us]
 5   session_number     435924 non-null  int64         
 6   registration_flag  435924 non-null  int64         
 7   page_counter       435924 non-null  int64         
 8   region             435924 non-null  str           
 9   device             435924 non-null  str           
dtypes: datetime64[us](3), int64(3), str(4)
memory usage: 33.3 MB


None

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
user_id,435924,134039,10E0DEFC1ABDBBE0,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
session_id,435924,435924,F9AF61A0C2023832,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
session_date,435924,NaN,NaN,NaN,2025-08-28 11:57:31.944834,2025-08-11 00:00:00,2025-08-21 00:00:00,2025-08-28 00:00:00,2025-09-05 00:00:00,2025-09-23 00:00:00,NaN
session_start_ts,435924,NaN,NaN,NaN,2025-08-29 05:24:32.054523,2025-08-11 00:00:25,2025-08-21 17:11:34.500000,2025-08-28 19:36:10,2025-09-05 17:58:17.250000,2025-09-23 23:51:46,NaN
install_date,435924,NaN,NaN,NaN,2025-08-25 04:43:14.433892,2025-08-11 00:00:00,2025-08-18 00:00:00,2025-08-25 00:00:00,2025-09-02 00:00:00,2025-09-10 00:00:00,NaN
session_number,435924.0,NaN,NaN,NaN,2.382908,1.0,1.0,2.0,3.0,10.0,1.288487
registration_flag,435924.0,NaN,NaN,NaN,0.077325,0.0,0.0,0.0,0.0,1.0,0.267107
page_counter,435924.0,NaN,NaN,NaN,3.005561,1.0,2.0,3.0,4.0,7.0,1.024625
region,435924,3,CIS,195975,NaN,NaN,NaN,NaN,NaN,NaN,NaN
device,435924,4,Android,195644,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Выведите первые пять строк датафрейма
sessions_history.head(5)

,user_id,session_id,session_date,session_start_ts,install_date,session_number,registration_flag,page_counter,region,device
0,E302123B7000BFE4,F9AF61A0C2023832,2025-08-15,2025-08-15 17:47:35,2025-08-15,1,0,3,CIS,iPhone
1,2530F72E221829FB,85003A206CBDAC6F,2025-08-15,2025-08-15 16:42:14,2025-08-15,1,0,4,MENA,Android
2,876E020A4FC512F5,3677423E49D72DEE,2025-08-15,2025-08-15 12:30:00,2025-08-15,1,0,4,EU,PC
3,2640B349E1D81584,956B45F5915CA225,2025-08-15,2025-08-15 15:31:31,2025-08-15,1,0,4,CIS,Android
4,94E1CBFAEF1F5EE9,83BF0DA35F9F1F40,2025-08-15,2025-08-15 21:33:53,2025-08-15,1,0,3,CIS,Android


### 1.2 Знакомство с данными

Основная метрика эксперимента — **доля успешных первых сессий**. Она рассчитывается с опорой на общие данные с результатами первых сессий пользователей.

Чтобы работать с историческими данными, сперва проверьте их структуру и подготовьте срез первых сессий:

- Убедитесь, что каждая строка соответствует отдельной сессии `session_id`.

- Создайте срез данных `first_sessions`, содержащий только первые сессии пользователей — то есть те, где `session_number == 1`.

- Удостоверьтесь, что каждому `user_id` соответствует не более одной первой сессии. Для этого выведите количество первых сессий и уникальных пользователей.

- Сформулируйте краткий вывод о структуре данных.

In [ ]:
# Проверьте, каждая ли строка соответствует отдельной сессии session_id
...

In [ ]:
# Создайте срез данных с первыми сессиями
first_sessions = ...

In [ ]:
# Сравните количество первых сессий и уникальных пользователей
# Проверьте, есть ли пользователи с несколькими первыми сессиями
...

### 1.3 Динамика числа первых сессий

Основная метрика рассчитывается по первым сессиям пользователей. Чтобы в дальнейшем оценить продолжительность теста, определите, сколько первых сессий появляется в приложении ежедневно. Для анализа используйте срез данных с первыми сессиями `first_sessions`.

- Рассчитайте количество первых сессий для каждого дня наблюдения.

- Постройте линейный график динамики первых сессий по дням.

- Определите период, за который в приложение поступили пользователи с первыми сессиями. Посчитайте количество дней в этом периоде.

- Рассчитайте среднее дневное число первых сессий за этот период. Результат сохраните в переменной `avg_first_sessions_daily`. Это значение понадобится для оценки длительности эксперимента.

- Проанализируйте результаты этого этапа работы и сформулируйте краткий вывод. Опишите динамику первых сессий и зафиксируйте среднее дневное значение.

In [ ]:
# Рассчитайте количество первых сессий по дням
first_sessions_daily = ...

In [ ]:
# Постройте график динамики первых сессий
plt.figure(figsize=(10, 5))

sns.lineplot(
    data=...,
    x='...',
    y='...',
    marker='o',
)

plt.title('...')
plt.xlabel('...')
plt.ylabel('...')
plt.grid()

plt.xticks(rotation=45)
plt.show()

In [ ]:
# Определите период поступления первых сессий
...

# Подсчитайте количество дней в периоде поступления первых сессий
first_sessions_total_days = ...
print(f'Всего дней: {first_sessions_total_days}')

# Подсчитайте среднее количество первых сессий в день за установленный период
avg_first_sessions_daily = ...
print(f'Среднее дневное число первых сессий за этот период: {avg_first_sessions_daily:1.0f}')

### 1.4 Анализ числа просмотренных страниц

Глубина просмотра — это признак, который показывает, сколько страниц пользователь просмотрел во время первой сессии. Сессия будет считаться успешной, только если глубина просмотра больше или равна 4, поэтому важно изучить этот показатель. Данные находятся в столбце `page_counter`.

Перед расчётом доли успешных сессий изучите распределение этого показателя:

- Рассчитайте количество первых сессий с разной глубиной просмотра. Рассмотрите каждое значение `page_counter`.

- Постройте столбчатую диаграмму, на которой по оси X отображено число просмотренных страниц, по оси Y — количество первых сессий.

- Опишите полученное распределение. Укажите, какие значения `page_counter` встречаются чаще всего и какая часть распределения соответствует успешным сессиям по условию `page_counter >= 4`.

In [ ]:
# Подсчитайте количество первых сессий с разным количеством просмотренных страниц
pages_scrolled = ...

# Визуализируйте результат: постройте столбчатую диаграмму
plt.figure(figsize=(10, 5))
...

# Оформление графика
plt.title('...')
plt.xlabel('...')
plt.ylabel('...')

plt.grid(axis='y')
plt.xticks(rotation=0)
plt.show()

### 1.5 Доля успешных первых сессий

Продуктовая команда считает первую сессию успешной, если пользователь просмотрел не менее 4 страниц (условие `page_counter >= 4`). Рассчитайте историческое значение доли успешных первых сессий и изучите динамику этой метрики.

- Создайте в датафрейме `first_sessions` столбец `good_session`. Поставьте в строки значения по следующему правилу: `1`, если пользователь просмотрел не менее 4 страниц, и `0` в остальных случаях.

- Рассчитайте долю успешных первых сессий за весь исторический период.

- Рассчитайте долю успешных первых сессий для каждого дня.

- Постройте график динамики доли успешных первых сессий по дням. Добавьте горизонтальную линию, показывающую среднее значение за весь период.

- Оцените, наблюдаются ли выраженный тренд или резкие изменения доли успешных первых сессий относительно значения метрики за весь исторический период.

In [ ]:
# Добавьте в датафрейм first_sessions признак good_session
...

In [ ]:
# Рассчитайте долю успешных первых сессий за весь исторический период
historical_good_session_rate = ...

print(f'Доля успешных первых сессий {historical_good_session_rate:0.3f}')

In [ ]:
# Рассчитайте долю успешных первых сессий по дням
good_sessions_per_day = ...


# Постройте график динамики доли успешных сессий по дням
plt.figure(figsize=(10, 5))
sns.lineplot(...)

# Добавьте на график линию, означающую долю успешных первых сессий за весь исторический период
plt.axhline(...)

# Оформление графика
plt.title('...')
plt.ylabel('...')
plt.xlabel('...')

plt.grid()
plt.legend()
plt.xticks(rotation=45)

plt.show()

## 2. Подготовка к тесту

Вы проанализировали исторические данные, познакомились с основной метрикой эксперимента, рассчитали её базовое значение и изучили её динамику. Теперь необходимо определить параметры планируемого A/B-теста.

Перед его запуском нужно:

- Зафиксировать основную и дополнительные метрики эксперимента.
- Определить минимальный значимый эффект.
- Зафиксировать направление статистической проверки.
- Рассчитать необходимый размер выборки.
- Оценить продолжительность теста.

### 2.1 Основная и дополнительные метрики эксперимента

Продуктовая команда хочет понять, помогает ли новый алгоритм рекомендаций сильнее вовлекать пользователей во время первого знакомства с приложением. Чтобы это проверить, нужна основная метрика эксперимента. Зафиксируем её:

**Основная метрика эксперимента** — доля успешных первых сессий среди всех первых сессий пользователей. Первая сессия считается успешной, если пользователь во время неё просмотрел не менее 4 страниц.

При этом во время оценки результатов A/B-теста важно учитывать не только основную метрику, но и дополнительные показатели:

- **Прокси-метрики** помогают подробнее оценить, как меняются поведение и вовлечённость пользователей.
- **Барьерные метрики** позволяют убедиться, что новый алгоритм не ухудшает другие важные показатели продукта.

Предложите подходящие прокси- и барьерные метрики для этого эксперимента и кратко объясните свой выбор. Рассуждайте максимально широко: вы можете предлагать и такие показатели, которых нет в имеющихся данных о проведении эксперимента. Вам нужно оценить, какие показатели теоретически полезны в поставленной задаче.

Рассчитывать и статистически проверять дополнительные метрики в рамках проекта не требуется.

### 2.2 Параметры планируемого A/B-теста

Продуктовая команда ожидает, что новый алгоритм увеличит долю успешных первых сессий. Относительный минимальный детектируемый эффект, на который они ориентируются, — это **рост основной метрики на 5%** от её исторического значения.

Относительный рост на 5% не означает увеличение метрики на 5 процентных пунктов. Абсолютный минимальный детектируемый эффект рассчитывается относительно базового значения:

$$
\mathrm{MDE_{abs}} = p_1 \times 0.05,
$$

где:

- $\mathrm{MDE_{abs}}$ — абсолютное изменение метрики, для обнаружения которого рассчитывается размер выборки при заданной мощности.
- $p_1$ — базовое значение основной метрики.
- $0.05$ — ожидаемый относительный рост метрики, выраженный в долях.

Ожидаемое значение метрики рассчитывается так:

$$
p_2 = p_1 + \mathrm{MDE_{abs}},
$$

где:

- $p_2$ — ожидаемое значение основной метрики после внедрения нового алгоритма.

Вы проанализировали исторические данные и знаете, что базовое значение основной метрики — около 0.31.

Ваши задачи:

1. Задайте относительный минимальный детектируемый эффект метрики в виде десятичной дроби.
2. Рассчитайте абсолютный минимальный детектируемый эффект метрики.
3. Рассчитайте ожидаемое значение основной метрики $p_2$.

При работе используйте код ниже.

In [ ]:
# Базовое значение основной метрики
p1 = 0.31

# Задайте относительный минимальный детектируемый эффект в виде десятичной дроби
relative_mde = ...

# Рассчитайте абсолютный минимальный детектируемый эффект основной метрики
absolute_mde = ...

# Рассчитайте ожидаемое значение основной метрики
p2 = ...

print(f'Базовое значение основной метрики: {p1:.2%}')
print(f'Заданный относительный минимальный детектируемый эффект: {relative_mde:.2%}')
print(f'Абсолютное изменение основной метрики: {absolute_mde * 100:.2f} п. п.')
print(f'Ожидаемое значение основной метрики: {p2:.2%}')

### 2.3 Направление статистической проверки

Продуктовая команда хочет проверить, увеличивает ли новый алгоритм основную метрику. Поэтому для проверки продуктовой гипотезы нужно использовать **односторонний статистический тест**.

Используемые в проекте функции для расчёта размера выборки и проведения статистического теста из пакета `statsmodels` имеют параметр `alternative`, который определяет направление проверки. Он может принимать следующие значения:

- `'two-sided'` — проверяет наличие любого различия между группами, независимо от его направления.
- `'larger'` — проверяет, превышает ли значение метрики в первой рассматриваемой группе значение во второй группе.
- `'smaller'` — проверяет, меньше ли значение метрики в первой рассматриваемой группе по сравнению со второй.

В эксперименте группа с новым алгоритмом будет сравниваться с контрольной группой. Поскольку команду интересует рост метрики в экспериментальной группе, зафиксируйте соответствующее значение параметра `alternative`.

При работе используйте код ниже.

In [ ]:
# Зафиксируйте направление статистической проверки
alternative = ...

### 2.4 Расчёт размера выборки одной группы эксперимента

В рамках курса вы уже рассчитывали размер выборки с помощью онлайн-калькулятора. В этом задании воспользуйтесь готовыми функциями и определите необходимое для проведения эксперимента количество пользователей.

Для расчёта задайте следующие параметры:

- уровень статистической значимости `alpha` — `0.05`;
- вероятность ошибки второго рода `beta` — `0.2`;
- мощность теста `power` — `1 - beta`.

Ваши задачи:

- С помощью функции `proportion_effectsize()` рассчитайте величину статистического эффекта. Передавая функции аргументы, учитывайте выбранное направление проверки: сначала укажите ожидаемое значение метрики (`p2`), затем базовое (`p1`).

- С помощью метода `solve_power()` класса `NormalIndPower` рассчитайте необходимый размер одной группы.

- Округлите полученное значение **вверх до целого числа**.

Решая задачу, используйте введённые ранее переменные с базовым значением метрики (`p1`), ожидаемым значением (`p2`) и направлением проверки (`alternative`).

Учитывайте, что результат соответствует необходимому количеству пользователей **в одной группе эксперимента**.

При работе используйте код ниже.

In [ ]:
# Задайте параметры теста
alpha = ...  # Уровень статистической значимости
beta = ...   # Вероятность ошибки второго рода
power = ...  # Мощность теста

# Рассчитайте величину статистического эффекта
effect_size = proportion_effectsize(..., ...)

# Инициализация класса NormalIndPower
power_analysis = NormalIndPower()

# Рассчитайте размер выборки для одной группы
sample_size = power_analysis.solve_power(
   effect_size=effect_size,
   power=power,
   alpha=alpha,
   ratio=1,
   alternative=alternative
)

# Округлите размер выборки вверх
sample_size = ...

# Вывод параметров планируемого теста
print(f'Базовое значение основной метрики, p1: {p1:.2%}')
print(f'Ожидаемое значение основной метрики, p2: {p2:.2%}')
print(f'Величина статистического эффекта: {effect_size:.4f}')
print(f'Уровень статистической значимости: {alpha:.2%}')
print(f'Мощность теста: {power:.2%}')
print(f'Направление статистической проверки: {alternative}')
print(f'Необходимый размер одной группы: {sample_size}')

### 2.5 Расчёт длительности A/B-теста

Зная минимально необходимое количество пользователей в одной группе, можно оценить продолжительность теста. Ранее вы рассчитали среднее дневное число первых сессий — `avg_first_sessions_daily`. Используйте это значение для расчёта длительности A/B-теста.

- В эксперименте участвуют две группы одинакового размера, поэтому сначала рассчитайте общее необходимое количество наблюдений.

- Разделите общий размер выборки на среднее дневное число первых сессий.

- Округлите полученное количество дней вверх до целого значения.

В эксперименте каждый новый пользователь учитывается по результату своей первой сессии, поэтому среднее дневное число первых сессий соответствует среднему числу новых пользователей, которых можно включить в тест.

При работе используйте код ниже.

In [ ]:
# Рассчитайте общий необходимый размер выборки для двух групп
total_sample_size = ...

# Рассчитайте длительность теста
test_duration = ...

# Округлите длительность теста вверх до целого числа дней
test_duration = ...

print(f'Необходимый размер одной группы: {sample_size}')
print(f'Общий необходимый размер выборки: {total_sample_size}')
print(f'Расчётная длительность теста: {test_duration} дней')

Проверьте результат. При текущих параметрах необходимый объём выборки можно набрать примерно за 6 дней. При этом активность пользователей может различаться в разные дни недели, поэтому продолжительность теста далее принимается равной 7 дням, чтобы охватить полный недельный цикл.

Выполнив все задания выше, напишите выводы, в которых зафиксируете основные параметры планируемого теста.

## 3. Мониторинг A/B-теста

### 3.1 Проверка распределения пользователей

A/B-тест успешно запущен, и уже доступны данные за первый день. На этом этапе нужно убедиться, что пользователи корректно распределены между группами.

Пользователь попадает в эксперимент во время своей первой сессии и закрепляется за одной из групп. Поскольку в данных могут встречаться повторные сессии, при проверке распределения учитывайте каждого пользователя только один раз.

Ваши задачи:

- Загрузите файл `sessions_project_test_part.csv` и сохраните данные в датафрейм `sessions_test_part`.

- Рассчитайте количество уникальных пользователей в каждой экспериментальной группе.

- Рассчитайте и выведите на экран процентную разницу в количестве пользователей в группах A и B. Постройте любую удобную визуализацию, на которой будет видно возможное различие между группами.

Для расчёта процентной разницы воспользуйтесь формулой:
$$P = 100 \cdot  \frac{|A − B|}{A},$$

где A и B — количество уникальных пользователей в соответствующих группах.

In [ ]:
# Загрузите данные за первый день теста и выполните парсинг колонок с датой и временем, указав их в аргументе parse_dates
sessions_test_part = pd.read_csv(
    DATA_PATH + "...",
    parse_dates=[...]
)

In [ ]:
# Выполните агрегацию уникальных пользователей по дням и группам
daily_users = ...

# Рассчитайте относительную разницу за первый день
diff_pct = abs(1 - daily_users.loc[:, 'B'] / daily_users.loc[:, 'A']).iloc[0] * 100
print(f"Разница в количестве пользователей: {diff_pct:.2f}%")

# Визуализация
...

### 3.2 Проверка пересечений пользователей
Убедитесь, что никто из пользователей случайно не попал в обе группы одновременно.

Рассчитайте количество пользователей, которые встречаются одновременно в группах A и B, или удостоверьтесь, что таких нет.

In [ ]:
# Проверка пересечений пользователей
users_a = ...
users_b = ...

print(f"Пользователей в контрольной группе {len(users_a)}.")
print(f"Пользователей в тестовой группе {len(users_b)}.")
print(f"Пересечение пользователей: {len(users_a & users_b)}")

### 3.3 Равномерность разделения пользователей по устройствам

Полезно также убедиться в том, что пользователи равномерно распределены по всем доступным категориальным переменным — типам устройств и регионам.

Постройте две диаграммы:

- Доля каждого типа устройства для пользователей из группы A.

- Доля каждого типа устройства для пользователей из группы B.

Добавьте на диаграммы все необходимые подписи, пояснения и заголовки, которые позволят сделать вывод о том, совпадает ли распределение устройств в группах A и B.

Каждый пользователь должен учитываться один раз. Доли рассчитывайте отдельно внутри каждой экспериментальной группы.

In [ ]:
# Диаграммы для проверки равномерности распределения пользователей по устройствам
...

### 3.4 Равномерность распределения пользователей по регионам

Теперь убедитесь, что пользователи равномерно распределены по регионам.

Постройте две диаграммы:

- Доля каждого региона для пользователей из группы A.

- Доля каждого региона для пользователей из группы B.

Добавьте на диаграммы все необходимые подписи, пояснения и заголовки, которые позволят сделать вывод о том, совпадает ли распределение регионов в группах A и B.

Каждый пользователь должен учитываться один раз. Доли рассчитывайте отдельно внутри каждой экспериментальной группы.

In [ ]:
# Диаграммы для проверки равномерности распределения пользователей по регионам
...

### 3.5 Выводы после проверки A/B-теста

На основе проведённого анализа A/B-теста сформулируйте и запишите свои выводы. Укажите:

- Было ли обнаружено различие в количестве пользователей в двух группах.

- Были ли обнаружены пользователи, попавшие одновременно в обе группы.

- Сохраняется ли равномерное распределение пользователей тестовой и контрольной групп по категориальным переменным — устройствам и регионам.

Сделайте выводы о том, корректно ли проходит A/B-тест. Определите, наблюдаются ли какие-то нарушения.

## 4. Проверка результатов A/B-теста

A/B-тест завершён. Полная выгрузка содержит первые и повторные сессии пользователей.

Эксперимент проводился 7 дней, с 14 по 20 октября 2025 года включительно. В выгрузке также присутствует небольшое число пользователей, чьи первые сессии произошли после завершения периода набора. Эти пользователи не входят в запланированный эксперимент и должны быть исключены из расчёта основной метрики.

Поэтому для анализа результатов используйте только первые сессии, произошедшие с 14 по 20 октября включительно.

### 4.1 Загрузка данных

- Загрузите файл `sessions_project_test.csv` и сохраните данные в датафрейм `sessions_test`.
- Выведите общую информацию о датафрейме и первые строки. Изучите структуру данных.
- Создайте датафрейм `test_first_sessions`, содержащий только первые сессии пользователей.
- Создайте столбец `good_session`. Заполните строки значениями `1`, если пользователь просмотрел не менее 4 страниц, и `0` в остальных случаях.

In [ ]:
# Загрузите данные теста
sessions_test = pd.read_csv(
    DATA_PATH + "...",
    parse_dates=[...]
)

In [ ]:
# Выведите информацию о загруженных данных
...

In [ ]:
# Выведите первые строки датафрейма sessions_test
...

In [ ]:
# Оставьте первые сессии пользователей
test_first_sessions = ...

In [ ]:
# Создайте признак успешной первой сессии
test_first_sessions['good_session'] = ...

### 4.2 Подготовка данных для анализа результатов теста

Прежде чем переходить непосредственно к результатам A/B-теста, нужно проверить период наблюдения, единицу анализа и фактический размер экспериментальных групп.

Ваши задачи:

- Рассчитайте количество первых сессий по дням.
- Проверьте, появляются ли первые сессии после 20 октября. Ограничьте данные периодом проведения эксперимента — с 14 по 20 октября включительно.
- Проверьте, что количество строк в `test_first_sessions` совпадает с количеством уникальных пользователей и количеством уникальных первых сессий.
- Рассчитайте количество уникальных пользователей в группах A и B и сравните фактическое число пользователей в каждой группе с рассчитанным значением `sample_size`.
- Сделайте выводы о периоде данных и о том, набран ли необходимый размер выборки в каждой группе.

In [ ]:
# Рассчитайте количество первых сессий по дням
first_sessions_daily_test = ...

In [ ]:
# Зафиксируйте период проведения эксперимента
test_start = ...
test_end = ...

# Ограничьте данные периодом проведения эксперимента
test_first_sessions = ...

In [ ]:
# Проверьте количество наблюдений
print('Количество строк:', ...)
print(
    'Количество уникальных пользователей:',
    ...
)
print(
    'Количество уникальных первых сессий:',
    ...
)

In [ ]:
# Рассчитайте фактический размер экспериментальных групп
sample_check = ...

In [ ]:
# Выведите следующую информацию
print(f'Минимальное количество пользователей в группе: {...}')
print(f'Минимальное необходимое количество пользователей в группе: {...}')

### 4.3 Формулировка нулевой и альтернативной гипотез

Перед тем как проводить A/B-тест, необходимо сформулировать нулевую и альтернативную гипотезы. На этапе планирования было зафиксировано, что продуктовую команду интересует рост основной метрики в группе с новым алгоритмом. Поэтому для анализа результатов используется односторонняя статистическая проверка с направлением `alternative='larger'`.

Сформулируйте нулевую и альтернативную гипотезы для сравнения доли успешных первых сессий в группах A и B.

Альтернативная гипотеза должна соответствовать росту основной метрики в группе B.

### 4.4 Сравнение доли успешных первых сессий

Перейдите к анализу основной метрики эксперимента — доли успешных первых сессий.

- Рассчитайте долю успешных первых сессий в группах A и B.
- Рассчитайте абсолютное изменение метрики в группе B относительно группы A в процентных пунктах.
- Рассчитайте относительное изменение метрики в группе B относительно группы A.
- Сравните полученное относительное изменение с ожидаемым ростом метрики в группе B на 5% относительно группы A.

In [ ]:
# Рассчитайте долю успешных первых сессий в группе A
good_sessions_rate_A = ...

# Рассчитайте долю успешных первых сессий в группе B
good_sessions_rate_B = ...

In [ ]:
# Рассчитайте абсолютное изменение в процентных пунктах
good_sessions_delta_pp = ...

# Рассчитайте относительное изменение метрики
good_sessions_relative_delta = ...

In [ ]:
print(f'Доля успешных первых сессий в группе A: {...}')
print(f'Доля успешных первых сессий в группе B: {...}')
print(f'Абсолютное изменение B-A: {...} п. п.')
print(f'Относительное изменение: {...}%')

### 4.5 Проверка статистической гипотезы

На предыдущем шаге доля успешных первых сессий в группе B оказалась немного ниже, чем в группе A. Однако описательного сравнения недостаточно для проверки статистической гипотезы. Для принятия решения всегда необходимо понять, является ли это изменение статистически значимым.

Ваши задачи:

- Используя односторонний z-тест для сравнения долей, проверьте гипотезу о росте основной метрики в группе B относительно группы A.

- Выведите значение p-value, сравните его с уровнем статистической значимости 0.05 и сформулируйте статистический вывод.

In [ ]:
# Односторонний z-тест для проверки роста метрики в группе B

# Количество успешных первых сессий
count = [...]

# Общее количество первых сессий
nobs = [...]

# Проведите односторонний z-тест
z_score, p_value = ...

print(f'p-value: {...}')

# Интерпретируйте результат
# Для этого заполните пробелы ниже
if p_value < alpha:
    print('...') # Напишите, что делать с нулевой гипотезой, если условие верно
    print('...') # Напишите, какие выводы из этого следуют

else:
    print('...') # Напишите, что делать с нулевой гипотезой, если условие ложно
    print('...') # Напишите, какие выводы из этого следуют

## 5. Выводы по результатам A/B-теста

На основе проведённого анализа сформулируйте выводы для продуктовой команды. Укажите:

- период и продолжительность теста;
- необходимое и фактическое число пользователей в каждой группе;
- значения основной метрики в группах A и B;
- абсолютное и относительное изменение метрики;
- значение p-value и статистический вывод;
- рекомендацию по внедрению нового алгоритма.